# R4-U0 - Protocol A WavLM embeddings the B cache lacks (Colab GPU)

Run on a **Colab GPU runtime** (the B extraction used a Tesla T4: 47,279 rows in about 23 minutes; this gap is 18,848 rows, so expect roughly 10 minutes plus shard staging). Run all cells top to bottom. Safe to rerun after a disconnect: finished shard packs are validated and reused.

**Extracts** only the A train crops and A validation chunks that `data/features/r2_u1_B` does not hold, listed by R4-S1 in `wavlm_cache_gap_A_v1.json` (7,222 Chatterbox train crops, 7,090 XTTS train crops, 4,536 Chatterbox validation chunks) into `data/features/r4_u0_A/`.

**Checks before publishing:** the gap is re-derived independently and must equal the file; the B cache and the new cache must cover every A train crop and A validation chunk exactly once (60,194 rows); 50 new rows are re-extracted and compared with the cache; **50 rows the B cache already holds are re-extracted and compared with B's stored vectors** (1e-4), so A and B embeddings can be used together.

**Never touches:** any A test recording (the plan contains none; test audio is not read), the B cache (read-only), any detector or score. No packages are installed and nothing is unzipped.

Requires `locks/B_lock_gr3.json` and the verified R4-S1 outputs on Drive.

In [ ]:
from pathlib import Path
import os, sys, json, time, hashlib, platform, subprocess, re, uuid, importlib.util
from datetime import datetime, timezone
from google.colab import drive

drive.mount("/content/drive")
ROOT = Path(os.environ.get("ECHOPROOF_ROOT", "/content/drive/MyDrive/PAI&AIES Project")).resolve()
assert (ROOT / "pyproject.toml").is_file(), ROOT
assert (ROOT / "locks/B_lock_gr3.json").exists(), "the GR3 lock is not on this Drive copy - Protocol A work must not start before it"
SCRATCH = Path("/content/echoproof-r4-u0").resolve()
assert SCRATCH.parent == Path("/content") and SCRATCH != ROOT
SCRATCH.mkdir(parents=True, exist_ok=True)
(SCRATCH / "tmp").mkdir(exist_ok=True)
os.environ.update(PYTHONDONTWRITEBYTECODE="1", TMPDIR=str(SCRATCH / "tmp"), TMP=str(SCRATCH / "tmp"), TEMP=str(SCRATCH / "tmp"))
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "src"))
BATCH_SIZE = 16     # the B extraction default
OUT = ROOT / "data/features/r4_u0_A"
REPORTS = ROOT / "reports/r4"
REPORTS.mkdir(parents=True, exist_ok=True)
print(json.dumps({"project": str(ROOT), "scratch": str(SCRATCH), "output": str(OUT)}, indent=2))

## 1. Check the prepared runtime (no installation)

The B cache was built with Transformers 5.22.0; a different version stops this notebook so the identity stays comparable.

In [ ]:
import ast
for source_path in list((ROOT / 'src/echoproof').rglob('*.py')) + [ROOT / 'scripts/r4_u0_wavlm_cache.py', ROOT / 'tests/test_r4_u0_wavlm_cache.py']:
    ast.parse(source_path.read_text(encoding='utf-8-sig'), filename=str(source_path))
required = ["numpy", "pandas", "pyarrow", "scipy", "sklearn", "soundfile", "librosa", "torch", "transformers", "huggingface_hub", "pytest"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
assert not missing, f"Missing dependencies: {missing}. Prepare the environment, restart, and rerun; nothing was installed."
import torch, transformers, numpy as np
assert torch.cuda.is_available(), "Choose Runtime > Change runtime type > GPU, then restart and rerun."
assert transformers.__version__ == "5.22.0", f"B cache used transformers==5.22.0; found {transformers.__version__}"
import echoproof
assert Path(echoproof.__file__).resolve().is_relative_to(ROOT), "Another EchoProof installation shadows this project."
spec = importlib.util.spec_from_file_location("r4_u0_runner", ROOT / "scripts/r4_u0_wavlm_cache.py")
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
from echoproof.features import r4_u0
from echoproof.artifacts import write_json, file_digest
SOURCE_HASHES = runner.source_hashes(ROOT)
ENVIRONMENT = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
               "gpu": torch.cuda.get_device_name(0), "gpu_vram_bytes": torch.cuda.get_device_properties(0).total_memory}
print(json.dumps(ENVIRONMENT, indent=2))

## 2. Metadata-only preflight

Checks the R4-S1 evidence chain (verified Colab report, manifest, input hashes, frozen A split hash, leakage report), the published B cache, and re-derives the gap independently. No audio is read and no model is downloaded.

In [ ]:
PLAN = r4_u0.preflight(ROOT)
print(json.dumps({"metadata_plan_sha256": PLAN.sha256, "counts": PLAN.counts, "coverage": PLAN.provenance["coverage"], "required_shards": PLAN.shards()}, indent=2))
assert PLAN.counts["logical_rows"] == 18848 and PLAN.provenance["coverage"]["a_rows"] == 60194, "unexpected plan size - stop and compare with the R4-S1 evidence"
assert set(PLAN.counts["by_partition"]) == {"train", "validation"}

## 3. Run the tests before reading research audio

Generated fixtures only. The report records the exit code, the pass count, skips and the hashes of the source files; the build refuses a report that does not match the current sources.

In [ ]:
test_command = [sys.executable, "-B", "-m", "pytest", "tests/test_r4_u0_wavlm_cache.py", "tests/test_r2_wavlm_cache.py", "tests/test_r1_u1_wavlm.py", "-q", "-ra", "-p", "no:cacheprovider"]
test_env = dict(os.environ, PYTHONPATH=str(ROOT / "src"), PYTHONDONTWRITEBYTECODE="1")
t0 = time.perf_counter()
completed = subprocess.run(test_command, cwd=ROOT, env=test_env, text=True, capture_output=True)
print(completed.stdout)
print(completed.stderr)
output = completed.stdout + "\n" + completed.stderr
passed = re.findall(r"(\d+) passed", output)
skipped = bool(re.search(r"\b\d+ skipped\b|\bSKIPPED\b", output))
TEST_REPORT = {"task": "R4-U0 fixture verification", "command": test_command, "returncode": completed.returncode,
               "passed_count": int(passed[-1]) if passed else 0, "skipped": skipped,
               "summary": output.strip().splitlines()[-1] if output.strip() else "", "stdout": completed.stdout, "stderr": completed.stderr,
               "seconds": round(time.perf_counter() - t0, 3), "source_sha256": SOURCE_HASHES, "environment": ENVIRONMENT}
TEST_PATH = REPORTS / ("r4_u0_tests_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8] + ".json")
write_json(TEST_PATH, TEST_REPORT)
assert runner.source_hashes(ROOT) == SOURCE_HASHES, "Source changed during tests; rerun from the top."
TESTS_PASSED = completed.returncode == 0 and TEST_REPORT["passed_count"] > 0 and not skipped
assert TESTS_PASSED, f"Tests did not pass without skips; inspect {TEST_PATH}. Audio has not been read."
print("Fixture tests passed; the GPU extraction and both parity checks are still pending.")

## 4. Stage the shards the gap needs on Colab scratch

Only the plan's shards are copied (checksummed, never overwritten). Shards also contain recordings outside the plan; extraction opens only the planned members, and no test member is planned.

In [ ]:
assert TESTS_PASSED
SHARDS = SCRATCH / "shards"
SHARDS.mkdir(exist_ok=True)
STAGING, stage_start = [], time.perf_counter()
for name in PLAN.shards():
    assert re.fullmatch(r"shard_\d+\.tar", name) and Path(name).name == name, name
    source, destination = ROOT / "data/processed/cohort_v1" / name, SHARDS / name
    assert source.is_file(), f"Missing canonical shard: {source}"
    if destination.exists():
        source_hash = file_digest(source)
        assert file_digest(destination) == source_hash, f"Staged shard differs: {destination}; inspect it before retrying."
        reused = True
    else:
        temporary = SHARDS / ("." + name + "." + uuid.uuid4().hex + ".partial")
        hasher = hashlib.sha256()
        try:
            with source.open("rb") as src, temporary.open("xb") as dst:
                for block in iter(lambda: src.read(1024 * 1024), b""):
                    hasher.update(block)
                    dst.write(block)
                dst.flush()
                os.fsync(dst.fileno())
            source_hash = hasher.hexdigest()
            assert file_digest(temporary) == source_hash, f"Copy checksum failed: {name}"
            assert not destination.exists(), f"Another writer created {destination}"
            temporary.replace(destination)
        finally:
            temporary.unlink(missing_ok=True)
        reused = False
    STAGING.append({"file": name, "sha256": source_hash, "bytes": destination.stat().st_size, "reused": reused})
    print(name, "reused" if reused else "copied", flush=True)
STAGING_SECONDS = round(time.perf_counter() - stage_start, 3)
print(json.dumps({"shards": len(STAGING), "bytes": sum(s["bytes"] for s in STAGING), "staging_seconds": STAGING_SECONDS}, indent=2))

## 5. Extract (or resume) and publish

The runner verifies the processor settings, model revision and weights, frozen eval mode and fp32, then extracts one pack per shard. The manifest is published last, only after the coverage check, cache integrity, the 50-row cached-versus-direct parity and the 50-row parity against B's stored vectors all pass. A failure leaves finished packs for `--resume` and publishes nothing.

In [ ]:
assert TESTS_PASSED and runner.source_hashes(ROOT) == SOURCE_HASHES
build_before = set(REPORTS.glob("r4_u0_build_*.json"))
command = [sys.executable, "-B", str(ROOT / "scripts/r4_u0_wavlm_cache.py"), "build", "--root", str(ROOT), "--shard-dir", str(SHARDS),
           "--model-cache", str(SCRATCH / "model-cache"), "--out-dir", str(OUT), "--batch-size", str(BATCH_SIZE), "--device", "cuda",
           "--resume", "--test-report", str(TEST_PATH)]
build_result = subprocess.run(command, cwd=ROOT, env=test_env)
new_reports = set(REPORTS.glob("r4_u0_build_*.json")) - build_before
assert len(new_reports) == 1, "Expected exactly one new build evidence report; inspect reports/r4."
BUILD_PATH = new_reports.pop()
BUILD_REPORT = json.loads(BUILD_PATH.read_text(encoding="utf-8"))
assert build_result.returncode == 0, f"Build failed; finished packs can be resumed. See {BUILD_PATH}"
assert BUILD_REPORT.get("verified") is True, f"GPU / coverage / parity verification did not pass: {BUILD_PATH}"
print("Build evidence:", BUILD_PATH)

## 6. Independent recheck and the hand-off report

Rereads the published cache, re-derives the plan, and reads the whole A union (B cache plus this cache) back through the combined reader. Share the resulting report and `data/features/r4_u0_A/r4_u0_manifest.json`. A verified extraction is not a trained model and not a teammate review.

In [ ]:
verify_before = set(REPORTS.glob("r4_u0_verify_*.json"))
verify_command = [sys.executable, "-B", str(ROOT / "scripts/r4_u0_wavlm_cache.py"), "verify", "--root", str(ROOT), "--out-dir", str(OUT)]
verify_result = subprocess.run(verify_command, cwd=ROOT, env=test_env)
new_reports = set(REPORTS.glob("r4_u0_verify_*.json")) - verify_before
assert len(new_reports) == 1, "Expected one independent verification report."
VERIFY_PATH = new_reports.pop()
VERIFY_REPORT = json.loads(VERIFY_PATH.read_text(encoding="utf-8"))
assert verify_result.returncode == 0, f"Independent integrity check failed: {VERIFY_PATH}"
assert runner.source_hashes(ROOT) == SOURCE_HASHES, "Source changed; do not accept this run."
MANIFEST = json.loads((OUT / "r4_u0_manifest.json").read_text(encoding="utf-8"))
FINAL_REPORT = {"task": "R4-U0", "created_utc": datetime.now(timezone.utc).isoformat(), "status": "colab_verified_pending_independent_review",
                "verified": True, "environment": ENVIRONMENT, "source_sha256": SOURCE_HASHES,
                "tests": {"file": str(TEST_PATH.relative_to(ROOT)), "sha256": file_digest(TEST_PATH), "summary": TEST_REPORT["summary"]},
                "build": {"file": str(BUILD_PATH.relative_to(ROOT)), "sha256": file_digest(BUILD_PATH)},
                "independent_verification": {"file": str(VERIFY_PATH.relative_to(ROOT)), "sha256": file_digest(VERIFY_PATH)},
                "staging": {"seconds": STAGING_SECONDS, "shards": STAGING}, "cache_directory": str(OUT.relative_to(ROOT)),
                "manifest_sha256": file_digest(OUT / "r4_u0_manifest.json"), "counts": MANIFEST["counts"], "coverage": MANIFEST["provenance"]["coverage"],
                "identity_vs_b": MANIFEST["identity_vs_b"],
                "parity": {"gap_rows": MANIFEST["parity"]["items"], "b_rows": MANIFEST["b_parity"]["items"],
                           "max_abs_gap": MANIFEST["parity"]["max_abs_embedding_difference"],
                           "max_abs_b": MANIFEST["b_parity"]["max_abs_embedding_difference"]},
                "extraction_seconds_all_packs": MANIFEST["extraction_seconds_all_packs"], "cache_bytes": MANIFEST["cache_bytes"],
                "test_rows_extracted": MANIFEST["test_rows_extracted"], "independent_reviewer": None, "trained_head_parity": "pending R4-U1"}
FINAL_PATH = REPORTS / ("r4_u0_colab_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8] + ".json")
write_json(FINAL_PATH, FINAL_REPORT)
print(json.dumps({"status": FINAL_REPORT["status"], "report": str(FINAL_PATH), "cache": str(OUT)}, indent=2))